# Robustness check: baseline at 4 vs 5 epochs (Option 3)

The baseline inherits its epoch count from `dual_view` (the only search). This
check shows that this transferred count does **not disadvantage** the baseline:
it additionally runs the baseline's 5-fold CV with the other epoch count(s)
from `config.ROBUSTNESS_EPOCHS` and compares **paired per fold** against the
canonical choice.

- Runs **only on the dev pool** - the test set is never touched.
- The canonical fold F1 are **reused** from `cv/baseline_v2/cv_summary.csv` (not
  recomputed); only the differing epoch count(s) cost compute time.
- **No** fold models are saved; the main outputs in `cv/` stay untouched.

**Decision rule (a priori):** if the mean fold difference is smaller than the
baseline's seed spread, the transfer is considered harmless. If a different
epoch count is clearly better, it becomes the reference (then recompute the
baseline CV with that epoch count so that `test_evaluation_v2` scores the right
fold models).

**Output:** `analysis/robustness_baseline_epochs*.csv`


In [ ]:
!pip install -q transformers sentencepiece accelerate scipy

In [ ]:
import re, json, os, random, sys
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import f1_score
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')
sys.path.insert(0, "/content/drive/MyDrive/google_colab/kusa/v2_heldout")
from config import *
import utils_split as u

VARIANT         = "baseline_v2"
BERT_MODEL_NAME = "xlm-roberta-large"
MAX_LEN         = 128
print("Variant:", VARIANT, "| getestete Epochenzahlen:", ROBUSTNESS_EPOCHS)

In [ ]:
# Fixed encoder hyperparameters from the (transferred) best_params.json.
with open(best_params_path(VARIANT), encoding="utf-8") as f:
    P = json.load(f)

BATCH_SIZE    = P["batch_size"]
LEARNING_RATE = P["lr"]
WEIGHT_DECAY  = P["weight_decay"]
WARMUP_RATIO  = P["warmup_ratio"]
CANON         = int(P["epochs"])          # the canonical, transferred epoch count

print(f"Fixed encoder HPs: batch={BATCH_SIZE}  lr={LEARNING_RATE}  "
      f"wd={WEIGHT_DECAY}  warmup={WARMUP_RATIO}")
print(f"Canonical epoch count (from HPO transfer): {CANON}")
assert CANON in ROBUSTNESS_EPOCHS, (
    f"config.ROBUSTNESS_EPOCHS {ROBUSTNESS_EPOCHS} should contain the canonical "
    f"epoch count {CANON} so a paired comparison is possible.")

In [ ]:
SEED = TRAIN_SEED
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# The same dev pool and the same folds as in the main CV.
cv_df = pd.read_csv(DEV_POOL, encoding="utf-8")
cv_df = cv_df.loc[:, ~cv_df.columns.str.contains("^Unnamed")]
cv_df = cv_df.dropna(subset=["surface"]).reset_index(drop=True)
folds_df = pd.read_csv(DEV_FOLDS, encoding="utf-8")
cv_df = cv_df.merge(folds_df, on="row_id", how="left", suffixes=("", "_f"))
if "fold_f" in cv_df.columns:
    cv_df = cv_df.drop(columns=["fold_f"])
assert cv_df["fold"].notna().all(), "rows without a fold assignment"
cv_df["fold"] = cv_df["fold"].astype(int)

with open(MANIFEST, encoding="utf-8") as f:
    MAN = json.load(f)
assert len(cv_df) == MAN["n_dev"], "development pool differs from the manifest"
print(f"Dev pool: {len(cv_df)} rows | fold sizes:",
      cv_df["fold"].value_counts().sort_index().tolist())

In [ ]:
# Dataset and training machinery - identical to kusa_baseline_cv_v2,
# but with epochs as a parameter instead of a fixed constant.
class SurfaceOnlyDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.texts  = df["surface"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[idx], dtype=torch.long)}


tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)


class CollapseError(RuntimeError):
    """Raised when a fine-tuning run diverges (epoch-1 loss above ln(3))."""


def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL_NAME, num_labels=3)


def make_loader(df, shuffle):
    return DataLoader(SurfaceOnlyDataset(df, tokenizer),
                      batch_size=BATCH_SIZE, shuffle=shuffle)


def train_model(train_df, seed, epochs, tag=""):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    train_loader = make_loader(train_df, shuffle=True)
    model = build_model(); model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                            weight_decay=WEIGHT_DECAY)
    total_steps  = len(train_loader) * epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"{tag}Epoch {epoch+1}/{epochs}", leave=False):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step(); scheduler.step()
            total_train_loss += loss.item()
        mean_loss = total_train_loss / len(train_loader)
        print(f"{tag}Epoch {epoch+1}/{epochs} | train loss: {mean_loss:.4f}")
        if epoch == 0 and mean_loss > COLLAPSE_LOSS:
            raise CollapseError(
                f"{tag}epoch-1 loss {mean_loss:.4f} > ln(3)={COLLAPSE_LOSS:.4f}")

    del optimizer, scheduler
    torch.cuda.empty_cache()
    return model


def train_model_guarded(train_df, seed, epochs, tag=""):
    for attempt in range(COLLAPSE_RETRIES):
        s = seed + 1000 * attempt
        try:
            return train_model(train_df, s, epochs, tag=tag), s
        except CollapseError as e:
            print(f"{tag}[collapse guard] {e} - retry {attempt+1}/{COLLAPSE_RETRIES}")
            torch.cuda.empty_cache()
    raise RuntimeError(f"{tag}training collapsed {COLLAPSE_RETRIES} times in a row")


def predict(model, df):
    loader = make_loader(df, shuffle=False)
    model.eval()
    preds, labels_out = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            labels_out.extend(labels.cpu().numpy())
    return np.array(preds), np.array(labels_out)

## Compute

The 5-fold CV per epoch count. The canonical count is reused from
`cv_summary.csv` (if present), the others are trained fresh - with the same fold
seeds `SEED + fold` and the same collapse guard as the main CV.


In [ ]:
cvsum_path = os.path.join(cv_dir(VARIANT), "cv_summary.csv")
REUSE_CANONICAL = True     # kanonische Fold-F1 aus cv_summary.csv wiederverwenden

results, fold_f1 = [], {}
for E in ROBUSTNESS_EPOCHS:
    print(f"\n{'='*60}\nEPOCHS = {E}\n{'='*60}")
    if E == CANON and REUSE_CANONICAL and os.path.exists(cvsum_path):
        s = pd.read_csv(cvsum_path).sort_values("fold")
        f1s = [float(x) for x in s["macro_f1"].tolist()]
        print(f"  Wiederverwendung von {len(f1s)} Fold-F1 aus cv_summary.csv")
    else:
        f1s = []
        for fold in range(N_FOLDS):
            tr = cv_df.iloc[np.where(cv_df["fold"].values != fold)[0]].reset_index(drop=True)
            va = cv_df.iloc[np.where(cv_df["fold"].values == fold)[0]].reset_index(drop=True)
            model, _ = train_model_guarded(tr, SEED + fold, E,
                                           tag=f"[{E}ep] Fold {fold+1} | ")
            preds, labels = predict(model, va)
            f1 = float(f1_score(labels, preds, average="macro"))
            f1s.append(f1)
            del model; torch.cuda.empty_cache()
            print(f"  epochs={E} Fold {fold+1}: macro-F1 {f1:.4f}")

    fold_f1[E] = np.array(f1s)
    for k, f in enumerate(f1s):
        results.append({"epochs": E, "fold": k + 1, "macro_f1": f})
    print(f"  -> mean {fold_f1[E].mean():.4f} +/- {fold_f1[E].std(ddof=1):.4f}")

## Comparison and verdict


In [ ]:
os.makedirs(ANALYSIS, exist_ok=True)

res_df = pd.DataFrame(results)
res_df.to_csv(os.path.join(ANALYSIS, "robustness_baseline_epochs.csv"),
              index=False, encoding="utf-8")

summ = res_df.groupby("epochs")["macro_f1"].agg(["mean", "std", "min", "max"]).round(4)
print("===== Summary per epoch count =====")
print(summ.to_string())
summ.to_csv(os.path.join(ANALYSIS, "robustness_baseline_epochs_summary.csv"),
            encoding="utf-8")

try:
    from scipy.stats import ttest_rel
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

base     = fold_f1[CANON]
base_std = float(base.std(ddof=1))
print(f"\nCanonical epochs={CANON}: mean {base.mean():.4f} +/- {base_std:.4f} "
      f"(seed spread)")

verdict_rows = []
for E in ROBUSTNESS_EPOCHS:
    if E == CANON:
        continue
    a = fold_f1[E]; d = a - base
    mean_d = float(d.mean())
    row = {"epochs": E, "vs_canonical": CANON,
           "mean_diff": round(mean_d, 4),
           "per_fold_diff": [round(float(x), 4) for x in d],
           "seed_std": round(base_std, 4),
           "within_seed_std": bool(abs(mean_d) < base_std)}
    if HAVE_SCIPY:
        t = ttest_rel(a, base)
        row["paired_t"] = round(float(t.statistic), 3)
        row["p"] = round(float(t.pvalue), 4)
    verdict_rows.append(row)

    print(f"\nepochs={E}  vs  canonical {CANON}")
    print(f"  difference per fold : {[round(float(x), 4) for x in d]}")
    print(f"  mean difference : {mean_d:+.4f}   (seed std {base_std:.4f})")
    if HAVE_SCIPY:
        print(f"  paired t-test   : t={row['paired_t']}, p={row['p']}")
    if abs(mean_d) < base_std:
        print(f"  -> |difference| < seed spread: the transferred epoch count "
              f"does NOT disadvantage the baseline. Transfer confirmed.")
    elif mean_d > 0:
        print(f"  -> epochs={E} is relevantly better. Consider baseline@{E} as "
              f"the reference and recompute the baseline CV with {E} epochs\n"
              f"     (regenerates the fold models that test_evaluation_v2 scores).")
    else:
        print(f"  -> epochs={E} is worse: the canonical choice {CANON} is "
              f"already the better one. No problem.")

pd.DataFrame(verdict_rows).to_csv(
    os.path.join(ANALYSIS, "robustness_baseline_epochs_verdict.csv"),
    index=False, encoding="utf-8")

assert_test_untouched(globals())
print("\nRobustness check complete - only dev pool used, test set untouched.")